# ML-07 — Baseline Action Score and Top-10 Review

**Lane:** Refresh / Content Opportunity Scoring (starter playground dataset).
Locking this lane for the baseline + Week-5 model — the w03 data-contract exploration on
`fact_content_daily_performance` (warehouse) stays as background context, but the flags this
assignment asks me to check (staleness, CTR-vs-position, volume) are the starter-CSV flags from
the session, so that's what this notebook runs on: `data/raw/content_refresh_anonymized.csv`.

Skills loaded: `building-baselines` (this task) + `flyrank/flyrank-data` (dataset gotchas).

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Two signal checks first**, then the rule they justify.

I lean on two signals from the session's real flags:
- **Staleness** — behind the `stale_visible_page` / refresh flags. Assumption: content that hasn't
  been touched in a long time is more likely to be declining.
- **Volume** — behind `is_quick_win` logic. Assumption: impression volume is a real proxy for how
  much opportunity is on the table, so it's worth gating on before spending review time.

I check both with a bucket table (n printed) against an outcome variable, **not** as inputs to the
score — `trend_direction` / `is_declining_label` are label-derived and are used here only to grade
the signal, never fed into the rule itself.

**The rule, in plain words:** *"A page is worth a human refresh review if it hasn't been touched
in 180+ days AND it's still pulling real search volume (≥500 impressions/90d)."* Both conditions
have to hold — staleness alone turned out to be a weak, non-monotonic signal on its own (see the
verdict below), so the rule doesn't fire on staleness alone. Volume is the one signal that held up
cleanly, so it does the heavy lifting: the score itself is just impressions, gated by the two
conditions being true. That's the same shape as the session's live example
(`stale * visible * impressions`) — readable on purpose, no fitted weights.

- **Reason code (one):** `stale_visible_page`
- **Action label:** `refresh` when the gate fires, `monitor` otherwise.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 120)

# Find the repo root from wherever this notebook runs (local clone, Colab, or CI) by walking up
# until data/raw/content_refresh_anonymized.csv is found.
def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw" / "content_refresh_anonymized.csv").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root (data/raw/content_refresh_anonymized.csv)")

REPO_ROOT = find_repo_root()
RAW_PATH = REPO_ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
df = pd.read_csv(RAW_PATH)
print(f"Loaded {len(df):,} rows x {df.shape[1]} columns")

# Reconstruct the label the same way scripts/01_prepare_features.py does, for GRADING the
# signals only. trend_direction / trend_pct / is_declining_label are NEVER used as rule inputs.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Base decline rate (whole dataset): {df['is_declining_label'].mean():.3f}  (n={len(df):,})")


Loaded 30,000 rows x 44 columns
Base decline rate (whole dataset): 0.542  (n=30,000)


In [2]:
# ---------------------------------------------------------------
# Signal check 1 — STALENESS, behind the refresh flags
# Assumption a refresh flag leans on: "old since last touched -> more likely declining"
# ---------------------------------------------------------------
stale = df["days_since_last_update"] >= 180

bucket1 = (
    df.assign(stale_bucket=np.where(stale, "stale_180plus", "not_stale"))
      .groupby("stale_bucket")
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .round(3)
)
print("Signal 1 -- staleness vs decline rate (binary)")
print(bucket1)
print()

# Finer view: freshness_tier already ships as a transparent bucket column
bucket1b = (
    df.groupby("freshness_tier")
      .agg(n=("content_id", "size"), decline_rate=("is_declining_label", "mean"))
      .reindex(["0-30", "31-90", "91-180", "181+"])
      .round(3)
)
print("Signal 1 -- by freshness_tier (finer buckets)")
print(bucket1b)
print()
print("VERDICT: MIXED")
print("- The stale bucket (n=174) actually shows a LOWER decline rate (0.471) than the")
print("  non-stale bucket (n=29,826, rate 0.542) -- the opposite of the naive assumption.")
print("- The finer freshness_tier view is non-monotonic: decline rate rises from 0-30 (0.511)")
print("  to 91-180 (0.611) then drops again at 181+ (0.471). Staleness alone does not track")
print("  decline cleanly in this slice, and the 181+ bucket is thin (n=174).")
print("- This is a real negative and it changes the rule: staleness alone is NOT a safe primary")
print("  driver of a score. It still gets used as a gate (see rule below), but it will not carry")
print("  the ranking on its own.")


Signal 1 -- staleness vs decline rate (binary)
                   n  decline_rate
stale_bucket                      
not_stale      29826         0.542
stale_180plus    174         0.471

Signal 1 -- by freshness_tier (finer buckets)
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471

VERDICT: MIXED
- The stale bucket (n=174) actually shows a LOWER decline rate (0.471) than the
  non-stale bucket (n=29,826, rate 0.542) -- the opposite of the naive assumption.
- The finer freshness_tier view is non-monotonic: decline rate rises from 0-30 (0.511)
  to 91-180 (0.611) then drops again at 181+ (0.471). Staleness alone does not track
  decline cleanly in this slice, and the 181+ bucket is thin (n=174).
- This is a real negative and it changes the rule: staleness alone is NOT a safe primary
  driver of a score. It still gets used 

In [3]:
# ---------------------------------------------------------------
# Signal check 2 -- VOLUME, behind the quick-win logic
# Assumption is_quick_win leans on: "impression volume is a real proxy for opportunity size"
# ---------------------------------------------------------------
order = ["low", "moderate", "good", "excellent"]  # no_data/none are empty in this slice, dropped
bucket2 = (
    df[df["impression_tier"].isin(order)]
      .groupby("impression_tier")
      .agg(n=("content_id", "size"), avg_clicks_90d=("clicks_90d", "mean"))
      .reindex(order)
      .round(2)
)
print("Signal 2 -- impression_tier vs average clicks_90d")
print(bucket2)
print()
print("VERDICT: CONFIRMED")
print("- avg_clicks_90d rises cleanly and monotonically with impression_tier: 0.24 -> 2.73 ->")
print("  30.42 -> 215.61 clicks, across large buckets (n from 1,078 to 11,248).")
print("- This confirms volume is a real, honest proxy for how much traffic is actually riding on")
print("  a page -- exactly the assumption is_quick_win leans on. Safe to gate the rule on it.")


Signal 2 -- impression_tier vs average clicks_90d
                     n  avg_clicks_90d
impression_tier                       
low              11248            0.24
moderate         10469            2.73
good              7205           30.42
excellent         1078          215.61

VERDICT: CONFIRMED
- avg_clicks_90d rises cleanly and monotonically with impression_tier: 0.24 -> 2.73 ->
  30.42 -> 215.61 clicks, across large buckets (n from 1,078 to 11,248).
- This confirms volume is a real, honest proxy for how much traffic is actually riding on
  a page -- exactly the assumption is_quick_win leans on. Safe to gate the rule on it.


## 2. Build the ranked queue (writes the CSV)

Score = `stale * visible * impressions_90d`. Both gates are binary (0/1); when either fails the
score is 0. Reason code and action label are attached the same way the session did it live.

In [4]:
stale_flag = (df["days_since_last_update"] >= 180).astype(int)
visible_flag = (df["impressions_90d"] >= 500).astype(int)

df["score"] = stale_flag * visible_flag * df["impressions_90d"]

df["reason_code"] = np.where(df["score"] > 0, "stale_visible_page", "no_flag")
df["action"] = np.where(df["score"] > 0, "refresh", "monitor")

queue = df.sort_values("score", ascending=False).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))

n_flagged = (queue["score"] > 0).sum()
print(f"Rows scored: {len(queue):,}   Rows flagged (score > 0): {n_flagged}")
print(f"Action counts:\n{queue['action'].value_counts()}")

output_cols = [
    "rank", "content_id", "client_id", "score", "reason_code", "action",
    "days_since_last_update", "impressions_90d", "clicks_90d", "avg_position",
    "ctr", "position_tier", "content_age_days", "word_count", "trend_direction",
]
output_path = REPO_ROOT / "work" / "outputs" / "baseline_action_score.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)
queue[output_cols].to_csv(output_path, index=False)
print(f"Wrote ranked queue -> {output_path}  ({len(queue):,} rows)")

queue[output_cols].head(10)


Rows scored: 30,000   Rows flagged (score > 0): 17
Action counts:
action
monitor    29983
refresh       17
Name: count, dtype: int64


Wrote ranked queue -> /home/claude/flyrank-ml-internship-starter/work/outputs/baseline_action_score.csv  (30,000 rows)


,rank,content_id,client_id,score,reason_code,action,days_since_last_update,impressions_90d,clicks_90d,avg_position,ctr,position_tier,content_age_days,word_count,trend_direction
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,stale_visible_page,refresh,194,61678,94,19.7,0.15,striking,231,5125.0,down
1,2,content_7368877ea310,client_7f2253d7e2,59472,stale_visible_page,refresh,194,59472,77,24.8,0.13,page_3_5,231,2591.0,down
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,stale_visible_page,refresh,194,25715,60,22.2,0.23,page_3_5,231,3861.0,down
3,4,content_0a91db491d14,client_7f2253d7e2,13299,stale_visible_page,refresh,193,13299,65,10.5,0.49,striking,231,3478.0,down
4,5,content_5feee3994adb,client_7f2253d7e2,7812,stale_visible_page,refresh,194,7812,1,39.0,0.01,page_3_5,231,3590.0,down
5,6,content_c2d929d83eaa,client_7f2253d7e2,7558,stale_visible_page,refresh,193,7558,15,17.9,0.20,striking,231,4758.0,down
6,7,content_b16bd7307b39,client_7f2253d7e2,4590,stale_visible_page,refresh,194,4590,0,31.0,0.00,page_3_5,231,4329.0,down
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,stale_visible_page,refresh,194,4556,15,16.4,0.33,striking,231,3388.0,down
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,stale_visible_page,refresh,194,4429,17,25.3,0.38,page_3_5,231,4486.0,down
9,10,content_928af3e22c80,client_7f2253d7e2,1697,stale_visible_page,refresh,193,1697,2,15.8,0.12,striking,231,3118.0,down


## 3. Top-10 review

For each of the top 10: the action, why it's there (in one line), and what single fact would
flip the call.

In [5]:
top10 = queue.head(10).reset_index(drop=True)
display(top10[["rank", "content_id", "client_id", "score", "action",
                "days_since_last_update", "impressions_90d", "avg_position", "ctr", "trend_direction"]])
print()

for _, r in top10.iterrows():
    why = (f"stale ({int(r['days_since_last_update'])}d since update) + visible "
           f"({int(r['impressions_90d']):,} impressions/90d) -> reason_code=stale_visible_page")
    wrong = (f"wrong if a refresh already shipped after export (days_since_last_update overstated), "
             f"or if the {int(r['impressions_90d']):,} impressions are mostly branded/irrelevant "
             f"queries the fix wouldn't touch")
    print(f"#{int(r['rank']):>2} [{r['action']}] {r['content_id']}")
    print(f"    why:   {why}")
    print(f"    wrong: {wrong}")
    print()


,rank,content_id,client_id,score,action,days_since_last_update,impressions_90d,avg_position,ctr,trend_direction
0,1,content_cf56e2e2e282,client_7f2253d7e2,61678,refresh,194,61678,19.7,0.15,down
1,2,content_7368877ea310,client_7f2253d7e2,59472,refresh,194,59472,24.8,0.13,down
2,3,content_1bfaa38ff26c,client_7f2253d7e2,25715,refresh,194,25715,22.2,0.23,down
3,4,content_0a91db491d14,client_7f2253d7e2,13299,refresh,193,13299,10.5,0.49,down
4,5,content_5feee3994adb,client_7f2253d7e2,7812,refresh,194,7812,39.0,0.01,down
5,6,content_c2d929d83eaa,client_7f2253d7e2,7558,refresh,193,7558,17.9,0.20,down
6,7,content_b16bd7307b39,client_7f2253d7e2,4590,refresh,194,4590,31.0,0.00,down
7,8,content_fe16a55cd13d,client_7f2253d7e2,4556,refresh,194,4556,16.4,0.33,down
8,9,content_ecb6215e79fd,client_7f2253d7e2,4429,refresh,194,4429,25.3,0.38,down
9,10,content_928af3e22c80,client_7f2253d7e2,1697,refresh,193,1697,15.8,0.12,down



# 1 [refresh] content_cf56e2e2e282
    why:   stale (194d since update) + visible (61,678 impressions/90d) -> reason_code=stale_visible_page
    wrong: wrong if a refresh already shipped after export (days_since_last_update overstated), or if the 61,678 impressions are mostly branded/irrelevant queries the fix wouldn't touch

# 2 [refresh] content_7368877ea310
    why:   stale (194d since update) + visible (59,472 impressions/90d) -> reason_code=stale_visible_page
    wrong: wrong if a refresh already shipped after export (days_since_last_update overstated), or if the 59,472 impressions are mostly branded/irrelevant queries the fix wouldn't touch

# 3 [refresh] content_1bfaa38ff26c
    why:   stale (194d since update) + visible (25,715 impressions/90d) -> reason_code=stale_visible_page
    wrong: wrong if a refresh already shipped after export (days_since_last_update overstated), or if the 25,715 impressions are mostly branded/irrelevant queries the fix wouldn't touch

# 4 [refresh] c

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [6]:
print("WEAK PICKS")
print("-" * 60)
client_counts = top10["client_id"].value_counts()
print(f"Top-10 client concentration:\n{client_counts}\n")
print("- All 10 of the top 10 (and 12 of the 17 total flagged rows) come from ONE client")
print("  (client_7f2253d7e2), all created on the same day (content_age_days = 231) and all last")
print("  touched ~193-194 days ago. This looks like a single content batch that shipped together")
print("  and was never revisited -- real, but it means the top of my queue isn't diversified across")
print("  clients. A rule this thin (n=17 flagged out of 30,000) would need per-client normalization")
print("  before it's fair to run across FlyRank's whole book.")
print("- Row content_5feee3994adb: 7,812 impressions but only 1 click (ctr=0.01%) and avg_position")
print("  39.0 -- this page may not be a 'refresh' fix at all; a position this deep with near-zero")
print("  CTR could mean the wrong keyword was targeted, not that the content went stale. Refreshing")
print("  text won't fix a targeting mismatch.")
print()

print("LEAKAGE CHECK")
print("-" * 60)
score_inputs = {"days_since_last_update", "impressions_90d"}
banned = {"trend_direction", "trend_pct", "is_declining_label",
          "health_score", "priority_score", "action_type", "refresh_tier"}
assert score_inputs.isdisjoint(banned), "score inputs accidentally overlap banned columns"
print(f"Score built only from: {sorted(score_inputs)}")
print(f"Confirmed NOT used as rule inputs (label-derived or product-flag columns): {sorted(banned & set(df.columns) | (banned - set(df.columns)))}")
print("- trend_direction / trend_pct / is_declining_label appear ONLY in the signal-check grading")
print("  (section 1) and in the CSV as read-only context columns -- never multiplied into `score`.")
print("- No FlyRank product flags (health_score, priority_score, action_type, refresh_tier) exist")
print("  in this dataset at all, so there was nothing to accidentally leak in.")
print("- No future window used: every column here is a trailing-90-day snapshot as of one export")
print("  date, not a forward-looking period.")


WEAK PICKS
------------------------------------------------------------
Top-10 client concentration:
client_id
client_7f2253d7e2    10
Name: count, dtype: int64

- All 10 of the top 10 (and 12 of the 17 total flagged rows) come from ONE client
  (client_7f2253d7e2), all created on the same day (content_age_days = 231) and all last
  touched ~193-194 days ago. This looks like a single content batch that shipped together
  and was never revisited -- real, but it means the top of my queue isn't diversified across
  clients. A rule this thin (n=17 flagged out of 30,000) would need per-client normalization
  before it's fair to run across FlyRank's whole book.
- Row content_5feee3994adb: 7,812 impressions but only 1 click (ctr=0.01%) and avg_position
  39.0 -- this page may not be a 'refresh' fix at all; a position this deep with near-zero
  CTR could mean the wrong keyword was targeted, not that the content went stale. Refreshing
  text won't fix a targeting mismatch.

LEAKAGE CHECK
------

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous `client_id`/`content_id`)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.